# Day 1 · Exercise 5: Audio to Video

**What you'll build:** `audio_to_video(audio_path, output_path, bg_color="0f172a", size=(1280,720))` — packages an audio file as a video with a solid colour background, matched exactly to the audio's duration.

**Why it matters:** Any pipeline that outputs audio needs to be able to package it as video — for platforms that require video format, for pipeline testing without real slides, or for generating placeholder frames. This function is also your first use of FFmpeg's `lavfi` virtual input, which generates synthetic media without any real source file.

**How to complete this exercise:**
1. Read the docstring — the two inputs and the key flags are specified there
2. Replace `pass` with your implementation
3. Run the **Check Your Work** cell — all 5 checks must pass

> **Hint if you're stuck:** You need two `-i` inputs: a lavfi color source and your audio file. Then `-shortest` to stop when the audio ends.

## Your Implementation

In [ ]:
import subprocess

def audio_to_video(
    audio_path: str,
    output_path: str,
    bg_color: str = "0f172a",
    size: tuple = (1280, 720),
) -> None:
    """
    Combine an audio file with a solid colour background to create a video.
    The output duration matches the audio duration exactly.

    Args:
        audio_path:  Path to the input audio file (MP3, WAV, AAC, etc.).
        output_path: Where to save the output MP4.
        bg_color:    Hex colour string WITHOUT the # prefix (default: '0f172a').
        size:        Output resolution as (width, height) tuple (default: (1280, 720)).

    Approach:
        First probe the audio's duration with ffprobe, then generate a colour
        video of exactly that length, so the two line up precisely.

    Key FFmpeg flags:
        -f lavfi                     Use the libavfilter virtual input device.
        -i "color=c=#{bg_color}:     Generate a solid colour video stream.
              s={w}x{h}:r=25:d={dur}" s = resolution, r = fps, d = duration in
                                     seconds — bounding it to the audio length.
        -i audio_path                The audio input.
        -c:v libx264                 H.264 video encoder.
        -pix_fmt yuv420p             Required for broadest device compatibility.
        -c:a aac -b:a 192k           AAC audio at 192 kbps.
        -shortest                    Safety net — stop at the shortest input.
        -y                           Overwrite output without prompting.

    Note: -shortest alone is NOT reliable against an *infinite* colour source
    (ffmpeg can overshoot the audio length), which is why we bound the colour
    source with d={dur} after probing the audio.

    Example:
        audio_to_video("lesson.mp3", "lesson_placeholder.mp4")
        audio_to_video("clip.mp3", "clip.mp4", bg_color="1e293b", size=(640, 360))
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    pass
    # ────────────────────────────────────────────────────────────────────────

## Check Your Work

Run the cell below. It generates a short test audio file, runs your function, and verifies the output has both a video stream and an audio stream with the correct duration.

In [ ]:
import os, subprocess, json

_PASS      = '\u2705'
_FAIL      = '\u274c'
_TEST_AUD  = '__check_audio.aac'
_TEST_VID  = '__check_atv.mp4'
_DURATION  = 3.0   # seconds
_W, _H     = 1280, 720

def _make_test_audio():
    """Generate a 3-second silent AAC file using FFmpeg."""
    r = subprocess.run(
        [
            "ffmpeg", "-y",
            "-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo",
            "-t", str(_DURATION),
            "-c:a", "aac",
            _TEST_AUD,
        ],
        capture_output=True,
    )
    return r.returncode == 0 and os.path.exists(_TEST_AUD)

def _ffprobe_streams(path):
    r = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", path],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        return None
    return json.loads(r.stdout).get("streams", [])

def _run_checks():
    score = 0
    total = 5

    if not _make_test_audio():
        print(f'{_FAIL} Could not generate test audio — is FFmpeg installed?')
        return

    # Check 1: function is callable
    try:
        assert callable(audio_to_video)
        print(f'{_PASS} Check 1/5: function exists and is callable')
        score += 1
    except AssertionError:
        print(f'{_FAIL} Check 1/5: audio_to_video is not defined')
        return

    # Check 2: creates a file
    try:
        audio_to_video(_TEST_AUD, _TEST_VID)
        assert os.path.exists(_TEST_VID), f'No file created at {_TEST_VID}'
        print(f'{_PASS} Check 2/5: creates a file at output_path')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 2/5: {e}')

    streams = _ffprobe_streams(_TEST_VID)

    # Check 3: has a video stream
    try:
        assert streams is not None, 'Could not read output file with ffprobe'
        video_streams = [s for s in streams if s['codec_type'] == 'video']
        assert video_streams, 'No video stream found in output'
        vs = video_streams[0]
        assert vs['width']  == _W, f"width: expected {_W}, got {vs['width']}"
        assert vs['height'] == _H, f"height: expected {_H}, got {vs['height']}"
        print(f"{_PASS} Check 3/5: has video stream at {vs['width']}x{vs['height']}")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 3/5: {e}')

    # Check 4: has an audio stream
    try:
        assert streams is not None
        audio_streams = [s for s in streams if s['codec_type'] == 'audio']
        assert audio_streams, (
            'No audio stream found in output — did you include the audio input '
            'and map it to the output?'
        )
        print(f"{_PASS} Check 4/5: has audio stream ({audio_streams[0]['codec_name']})")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 4/5: {e}')

    # Check 5: duration matches audio (within 1s)
    try:
        assert streams is not None
        # Use format duration for overall file length
        r2 = subprocess.run(
            ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
             "-of", "csv=p=0", _TEST_VID],
            capture_output=True, text=True,
        )
        dur = float(r2.stdout.strip())
        assert abs(dur - _DURATION) < 1.0, (
            f"duration: expected ~{_DURATION}s, got {dur:.2f}s — "
            f"did you use -shortest? Without it the colour source runs forever."
        )
        print(f"{_PASS} Check 5/5: duration matches audio ({dur:.2f}s)")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 5/5: {e}')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 5 complete! All {total}/{total} checks passed.')
        print('  You have finished all Day 1 exercises.')
        print('  Open project.ipynb to start the Day Project.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} checks passed. Keep going — you\'re close!')

    for f in [_TEST_AUD, _TEST_VID]:
        if os.path.exists(f):
            os.remove(f)

_run_checks()

## Bonus Challenge

Once all 5 checks pass, extend `audio_to_video` to accept an optional `title_text` parameter. If provided, use FFmpeg's `drawtext` filter to render the text centered on the background:

```python
# Add to your filter: drawtext=text='My Lesson':fontcolor=white:fontsize=48:x=(w-text_w)/2:y=(h-text_h)/2
# This requires FFmpeg to be compiled with --enable-libfreetype
# Check first: ffmpeg -filters 2>&1 | grep drawtext
```

If `drawtext` isn't available (it wasn't in our environment — which is why `slide_gen.py` uses Pillow instead), fall back to the plain colour background. This is a real trade-off you'll encounter: FFmpeg text rendering requires a compiled-in library, while Pillow works anywhere Python does.

In [ ]:
# Bonus: try audio_to_video with a real MP3 from earlier exercises
# (assuming you completed the Day 1 project or Exercise 1)

# import glob
# mp3s = glob.glob("briefing_*.mp3") + glob.glob("*.mp3")
# if mp3s:
#     audio_to_video(mp3s[0], "my_audio_video.mp4")
#     print(f"Created my_audio_video.mp4 from {mp3s[0]}")
# else:
#     print("No MP3 found — run Exercise 1 or the Day Project first")

---
## Solution

<details>
<summary>Click to reveal solution — try on your own first</summary>

```python
def audio_to_video(
    audio_path: str,
    output_path: str,
    bg_color: str = "0f172a",
    size: tuple = (1280, 720),
) -> None:
    w, h = size

    # 1. Probe the audio duration so the background lasts exactly as long.
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
         "-of", "csv=p=0", audio_path],
        capture_output=True, text=True, check=True,
    )
    duration = float(probe.stdout.strip())

    # 2. Generate a FINITE colour video of that exact length, then mux the audio.
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-f", "lavfi",
            "-i", f"color=c=#{bg_color}:s={w}x{h}:r=25:d={duration}",
            "-i", audio_path,
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-c:a", "aac",
            "-b:a", "192k",
            "-shortest",
            output_path,
        ],
        capture_output=True,
        check=True,
    )
```

**Why `-shortest` matters:**  
The lavfi `color` source generates video indefinitely — it has no natural end point. Without `-shortest`, FFmpeg would wait for the shortest input to end... except the color source never ends, so it defaults to the audio length anyway on recent FFmpeg versions. Using `-shortest` makes the intent explicit and is more reliable across FFmpeg versions.

**Why `lavfi` instead of a PNG:**  
`-f lavfi -i "color=..."` generates the colour frame synthetically without creating any files. This is faster than generating a PNG with Pillow and passing it as an input. The result is identical — a solid colour background at the specified resolution.

**Why this exercise ties Day 1 together:**  
- You're calling FFmpeg via subprocess (Exercise 4 pattern)
- You're verifying the output with ffprobe (Exercise 3 pattern)
- The function produces media that needs both a video and audio stream — the same structure as the final lesson video
- The `lavfi` source is how `lesson_build.py` could generate a placeholder video for a lesson with no talking head yet

</details>